> **[CAPSTONE INTEGRATION TARGET]** Plato receives untrusted physical-pipeline data. A narrow contextBridge grants only snapshot operations; Main authorizes the frame and validates an exact payload before exposing telemetry to the Renderer.

# Phase 2 — Validate data, authorize callers, keep text inert

Tasks: VF-200, VF-201, VF-202. Default execution uses a Python teaching fixture aligned with the application's request and telemetry contracts. Run `npm test` for actual TypeScript contract tests and the documented Electron smoke test for the runtime boundary.


In [ ]:
from pathlib import Path
import sys, os, json, asyncio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts" / "notebook_support.py").exists()), None)
assert ROOT is not None, "Open this notebook inside the supplied VoltForge source bundle"
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import notebook_support as lab
LIVE = os.environ.get("VOLTFORGE_LIVE_MCP") == "1"
print(json.dumps({"event": "lab.mode", "live_opt_in": LIVE, "workspace": ROOT.name}))


## The complete public API

```ts
window.plato.getSnapshot(): Promise<Result<Snapshot>>;
window.plato.onSnapshot(callback: (snapshot: Snapshot) => void): () => void;
```

Preload creates the internal `{requestId}`; Renderer supplies no channel, filesystem path, JavaScript source, or OS command. Internally, `plato:snapshot` accepts exactly one requestId matching `[A-Za-z0-9_-]{1,64}`. Main independently checks the trusted top-level frame. Schema validation is not a general JavaScript sanitizer.

Prediction: Should `{requestId:'ok', command:'readFile'}` pass merely because `requestId` is valid?


In [ ]:
accepted = lab.validate_request({"requestId": "plato-request-001"})
assert accepted == {"requestId": "plato-request-001"}
allowed = await lab.invoke_snapshot(accepted, trusted=True)
assert allowed["ok"] is True
print(json.dumps(allowed, indent=2))


## Falsification matrix: reject without executing

Every sample below is inert test data. We never call `eval`, run a shell command, or evaluate a payload in Electron. Prototype-like and unknown keys are rejected because the contract has exact fields.


In [ ]:
cases = {
    "unknown authority": {"requestId": "valid", "command": "readFile"},
    "code string": {"requestId": "require('fs').readFileSync('/etc/passwd')"},
    "HTML string": {"requestId": "<script>alert(1)</script>"},
    "wrong type": {"requestId": 42},
    "empty identifier": {"requestId": ""},
    "oversized identifier": {"requestId": "a" * 65},
    "prototype-like key": {"requestId": "valid", "__proto__": {"privileged": True}},
    "array shape": ["valid"],
    "missing payload": None,
}
audit = []
for label, payload in cases.items():
    result = await lab.invoke_snapshot(payload)
    assert result["ok"] is False and result["error"]["code"] == "INVALID_REQUEST", label
    audit.append({"case": label, "expected": "INVALID_REQUEST", "observed": result["error"]["code"]})
print(json.dumps(audit, indent=2))


## Correct data does not create authority

An embedded or foreign frame must fail even when it supplies valid data. The fixture's `trusted` switch models this independent check; the native handler derives it from the actual invoking frame URL and identity. Renderer cannot submit a flag to declare itself trusted.


In [ ]:
unauthorized = await lab.invoke_snapshot({"requestId": "shape-is-valid"}, trusted=False)
assert unauthorized["error"]["code"] == "UNTRUSTED_SENDER"
print(json.dumps(unauthorized))


## Validate telemetry before it reaches the UI

Telemetry version 1 uses safe integer sequence/timestamps, a bounded source name, at most 64 uniquely identified tracks, normalized x/y in [-1,1], confidence in [0,1], and exact hardware status values. Main separately enforces network byte/rate limits; this fixture validates decoded payloads.


In [ ]:
import copy
packet = lab.example_packet()
assert lab.validate_telemetry(packet) == packet
bad_packets = []
for label, mutate in [
    ("non-finite x", lambda p: p["objects"][0].update(x=float("nan"))),
    ("unknown version", lambda p: p.update(version=2)),
    ("out-of-range confidence", lambda p: p["objects"][0].update(confidence=2)),
    ("duplicate track id", lambda p: p["objects"].append(dict(p["objects"][0]))),
    ("unknown camera state", lambda p: p["hardware"].update(camera="probably-ready")),
    ("boolean sequence", lambda p: p.update(sequence=True)),
]:
    malformed = copy.deepcopy(packet)
    mutate(malformed)
    try:
        lab.validate_telemetry(malformed)
        raise AssertionError(f"Unexpected acceptance: {label}")
    except lab.ValidationError as error:
        bad_packets.append({"case": label, "rejected": True, "reason": str(error)})
print(json.dumps(bad_packets, indent=2))


## Native verification and optional live bridge inspection

For this phase use a separately reviewed MCP config whose Renderer mapping evaluates a read-only expression such as `({ api: Object.keys(window.plato), requireType: typeof require, processType: typeof process, snapshot: await window.plato.getSnapshot() })`, adapted to the discovered evaluation tool. The public API does not allow forging `requestId`; raw payload/sender tests belong to Main's contract tests. With electron-driver, keep the standard owned-session spawn/Main/stop mappings. Set the config before this notebook alone; don't reuse a DOM-mutation mapping and call it a security audit.


In [ ]:
if LIVE:
    config_path = os.environ.get("VOLTFORGE_MCP_CONFIG")
    assert config_path, "A reviewed phase-specific MCP config is required"
    native = await lab.live_mcp(lab.load_live_config(config_path))
    print(json.dumps(native, indent=2))
else:
    print(json.dumps({"event": "live.skipped", "phase": "security", "reason": "explicit opt-in absent"}))


## Explain the four independent controls

Fill in one sentence and one native evidence item for each: (1) schema validation, (2) sender authorization, (3) context isolation/sandbox, and (4) content security policy plus safe text sinks. None substitutes for the others. Run the bridge challenge at `../docs/phase-2/index.html`, whose display-only schema is intentionally smaller; the app's real API remains the contract above.

Acceptance: all invalid fixture cases rejected, valid case accepted, no attacker text evaluated, actual TypeScript tests pass, native inspection is attached or marked `not run`. Continue with Phase 3.
